In [2]:
import warnings
warnings.filterwarnings("ignore")
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware


### Data Ingestion

In [3]:
def data_ingestion(sample_txt_file):

    # Text Loader
    from langchain_community.document_loaders import TextLoader
    text_loader = TextLoader(sample_txt_file, encoding="utf-8")
    text_loader
    # Load the document
    document = text_loader.load() # Gives the document structure with page_content and updated metadata
    print(type(document))
    return document
    



# Creating a txt file with the content of the document
os.makedirs("../data/test_files", exist_ok=True)
sample_txt_file = "../data/bridge_health_report.txt"
doc = data_ingestion(sample_txt_file)

<class 'list'>


### Chunking


In [4]:
def chunk(document , chunk_size = 500 , chunk_overlap = 50):
    # Split for better RAG performance
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size , chunk_overlap=chunk_overlap , length_function=len,separators=["---", "\n\n", "\n", ". ", " "])
    split_doc = text_splitter.split_documents(document) # split_documents takes a list of documents and splits each document into smaller chunks based on the specified chunk_size and chunk_overlap. The resulting split_doc is a list of smaller Document objects, each containing a portion of the original document's content and metadata.
    print(f"Number of chunks created: {len(split_doc)}")  
    if split_doc:
        print("Example chunk:")
        print(f"Chunk content: {split_doc[0].page_content}") # page_content contains the actual text content of the chunk, which is a portion of the original document's content. This is the part that will be used for processing and retrieval in RAG.
        print(f"Chunk metadata: {split_doc[0].metadata}") # metadata contains the additional information about the chunk, such as its source, author, and creation date. This metadata is inherited from the original document and can be useful for organizing and retrieving chunks later on.
    return split_doc

chunks = chunk(doc)
chunks
        


Number of chunks created: 51
Example chunk:
Chunk content: BRIDGE HEALTH MONITORING REPORT
Generated   : 2026-04-10 16:35:58.505000
Total Windows : 50
Health State Distribution:
health_state
Healthy    37
Warning    12
Watch       1
Chunk metadata: {'source': '../data/bridge_health_report.txt'}


[Document(metadata={'source': '../data/bridge_health_report.txt'}, page_content='BRIDGE HEALTH MONITORING REPORT\nGenerated   : 2026-04-10 16:35:58.505000\nTotal Windows : 50\nHealth State Distribution:\nhealth_state\nHealthy    37\nWarning    12\nWatch       1\n========================================'),
 Document(metadata={'source': '../data/bridge_health_report.txt'}, page_content="During monitoring window 1 recorded at 2026-04-10 16:30:22.492000, the bridge was classified in a 'Watch' health state. The physical sensors recorded a filtered strain of 71.9253 mV, a deck distance of 5.8943 cm, and a total vibration of 0.940324 g RMS. Anomaly detection systems flagged acceleration as False, strain as False, and distance as True."),
 Document(metadata={'source': '../data/bridge_health_report.txt'}, page_content="---\nDuring monitoring window 2 recorded at 2026-04-10 16:30:29.668000, the bridge was classified in a 'Healthy' health state. The physical sensors recorded a filtered strain of 

### Embedding 

In [5]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9119.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### VectorDB

In [7]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "bridge_sensor_data", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    


Vector store initialized. Collection: bridge_sensor_data
Existing documents in collection: 153


In [8]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 51 texts...


Batches: 100%|██████████| 2/2 [00:00<00:00,  5.15it/s]

Generated embeddings with shape: (51, 384)
Adding 51 documents to vector store...
Successfully added 51 documents to vector store
Total documents in collection: 204


### RAG Retrieval

In [10]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

rag_retriever

rag_retriever.retrieve("What is a bridge health state?")

Retrieving documents for query: 'What is a bridge health state?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 91.42it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_f7d3bfda_0',
  'content': 'BRIDGE HEALTH MONITORING REPORT\nGenerated   : 2026-04-10 16:35:58.505000\nTotal Windows : 50\nHealth State Distribution:\nhealth_state\nHealthy    37\nWarning    12\nWatch       1\n========================================',
  'metadata': {'content_length': 214,
   'doc_index': 0,
   'source': '../data/bridge_health_report.txt'},
  'similarity_score': 0.3078792095184326,
  'distance': 0.6921207904815674,
  'rank': 1},
 {'id': 'doc_2675f3a9_0',
  'content': 'BRIDGE HEALTH MONITORING REPORT\nGenerated   : 2026-04-10 16:35:58.505000\nTotal Windows : 50\nHealth State Distribution:\nhealth_state\nHealthy    37\nWarning    12\nWatch       1\n========================================',
  'metadata': {'doc_index': 0,
   'source': '../data/bridge_health_report.txt',
   'content_length': 214},
  'similarity_score': 0.3078792095184326,
  'distance': 0.6921207904815674,
  'rank': 2},
 {'id': 'doc_4f2dbcfa_0',
  'content': 'BRIDGE HEALTH MONITORING REPORT\nGen

### Integrating VectorDB Context Pipeline with LLM output

In [13]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""You are BridgeAI, an AI assistant exclusively dedicated to bridge structural health monitoring and engineering.

    Your knowledge scope is strictly limited to:
    - This bridge's live sensor data and health reports
    - General politeness and professionalism in communication
    - Bridge engineering, design, and structural analysis
    - Vibration analysis, fatigue, and predictive maintenance
    - Civil and structural engineering principles related to bridges

    You are NOT permitted to answer questions outside this scope under any circumstances. Do not attempt to be helpful on out-of-scope topics. Do not provide partial answers. Do not acknowledge the out-of-scope topic at all.

    Here is the current live data and report context for this specific bridge:
    {context}

    Question: {query}

    Instructions:
    1. If the question is about the current status, health scores, or recent reports, answer strictly using the provided context.
    2. If the question is a general bridge or structural engineering question, use your structural engineering knowledge to answer it.
    3. If combining both, clearly state what is from the actual bridge data versus what is a general engineering principle.
    4. If the question is outside your defined scope — meaning it is not about bridges, structural engineering, or this bridge's health data — respond with exactly this and nothing else: "I can only assist with bridge health monitoring and structural engineering questions. This question is outside my scope."

    Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [24]:
answer=rag_simple(" what is the current health status of bridge?",rag_retriever,llm)
print(answer)

Retrieving documents for query: ' what is the current health status of bridge?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 93.83it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Based on the provided bridge health monitoring reports, the current health status of the bridge is as follows:

- Total Windows: 50
- Health State Distribution:
  - Healthy: 37
  - Warning: 12
  - Watch: 1

This indicates that 74% of the bridge's health monitoring windows are in a healthy state, 24% are in a warning state, and 2% are in a watch state.
